# RAA End-to-End Test Notebook

Tests the **current** RAA implementation by:
1. Loading raw ARLO output from `result.pkl` and original requirements from `requirements.json`
2. Applying the Orchestrator transformation (PRD v1 §3.2): enrich → strip → reshape
3. Invoking `RAA().run()` with the produced `RAAInput`
4. Inspecting the `RAAOutput`

In [1]:
from __future__ import annotations

import json
import pickle
import os
import uuid
from pathlib import Path
from IPython.display import JSON, Markdown, display

In [2]:
import sys                                                                                                                                                                                  
for mod in list(sys.modules.keys()):                                                                                                                                                        
  if mod.startswith("raa"):                                                                                                                                                               
      del sys.modules[mod]

In [3]:
# Kernel diagnostic — run this FIRST to trace the issue
import sys
print("Python executable:", sys.executable)
print("sys.path:")
for p in sys.path:
    print("  ", p)
print()
try:
    import raa
    print("raa imported OK from:", raa.__file__)
except ModuleNotFoundError as e:
    print("FAILED:", e)
    # Check if editable install exists
    import subprocess
    result = subprocess.run([sys.executable, "-m", "pip", "show", "I-Architect"], capture_output=True, text=True)
    print("pip show I-Architect:")
    print(result.stdout or result.stderr)


Python executable: /home/delatom/I-Architect-3cf20c60d77417e9febe099eeb91bc78227ce89f/.venv/bin/python
sys.path:
   /usr/lib/python312.zip
   /usr/lib/python3.12
   /usr/lib/python3.12/lib-dynload
   
   /home/delatom/I-Architect-3cf20c60d77417e9febe099eeb91bc78227ce89f/.venv/lib/python3.12/site-packages
   __editable__.i_architect-1.0.finder.__path_hook__

raa imported OK from: /home/delatom/I-Architect-3cf20c60d77417e9febe099eeb91bc78227ce89f/raa/__init__.py


## 1. Paths & Configuration

In [4]:
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "raa").exists() and (REPO_ROOT.parent / "raa").exists():
    REPO_ROOT = REPO_ROOT.parent

REQUIREMENTS_PATH = REPO_ROOT / "data" / "requirements.json"
PICKLE_PATH = REPO_ROOT / "test_project" / "result.pkl"

# Fresh thread id per run to avoid stale LangGraph checkpoints
THREAD_ID = f"notebook-raa-test-{uuid.uuid4().hex[:8]}"
DB_PATH = str(REPO_ROOT / "checkpoints" / "raa_test.db")
Path(REPO_ROOT / "checkpoints").mkdir(parents=True, exist_ok=True)

display(Markdown(f"**Repo Root:** `{REPO_ROOT}`"))
display(Markdown(f"**Requirements:** `{REQUIREMENTS_PATH}`"))
display(Markdown(f"**Pickle:** `{PICKLE_PATH}`"))
display(Markdown(f"**Thread ID:** `{THREAD_ID}`"))

**Repo Root:** `/home/delatom/I-Architect-3cf20c60d77417e9febe099eeb91bc78227ce89f`

**Requirements:** `/home/delatom/I-Architect-3cf20c60d77417e9febe099eeb91bc78227ce89f/data/requirements.json`

**Pickle:** `/home/delatom/I-Architect-3cf20c60d77417e9febe099eeb91bc78227ce89f/test_project/result.pkl`

**Thread ID:** `notebook-raa-test-51eaf518`

## 2. LLM Initialization

Requires `langchain-anthropic` (install via `pip install 'langchain-anthropic>=1.4'`).
Uses DeepSeek via the Anthropic-compatible endpoint.

In [5]:
from langchain_anthropic import ChatAnthropic

DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY")
if not DEEPSEEK_API_KEY:
    raise RuntimeError("Set DEEPSEEK_API_KEY in your environment before running.")

llm = ChatAnthropic(
    model="deepseek-v4-flash",
    temperature=0.0,
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com/anthropic",
    model_kwargs={"thinking": {"type": "disabled"}},
)
display(Markdown("**DeepSeek LLM initialized.**"))

/home/delatom/I-Architect-3cf20c60d77417e9febe099eeb91bc78227ce89f/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3688: UserWarning: Parameters {'thinking'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


**DeepSeek LLM initialized.**

## 3. Load Raw ARLO Output & Requirements

In [6]:
if not REQUIREMENTS_PATH.exists():
    raise FileNotFoundError(f"Requirements file not found: {REQUIREMENTS_PATH}")
if not PICKLE_PATH.exists():
    raise FileNotFoundError(f"ARLO pickle not found: {PICKLE_PATH}")

requirements = json.loads(REQUIREMENTS_PATH.read_text(encoding="utf-8"))

with open(PICKLE_PATH, "rb") as f:
    arlo_output = pickle.load(f)

display(Markdown("### Raw ARLO Output Summary"))
display(JSON({
    "requirements_count": len(requirements),
    "arlo_keys": list(arlo_output.keys()),
    "asrs_count": len(arlo_output.get("asrs", [])),
    "non_asr_count": len(arlo_output.get("non_asr", [])),
    "condition_groups_count": len(arlo_output.get("condition_groups", [])),
    "concerns_count": len(arlo_output.get("concerns", [])),
    "quality_weights": arlo_output.get("quality_weights", {}),
}))

### Raw ARLO Output Summary

<IPython.core.display.JSON object>

## 4. Orchestrator Transformation (PRD v1 §3.2)

Three sub-steps:
1. **Requirement Text Enrichment** — hydrate requirement IDs with full text from `requirements.json`
2. **Field Stripping** — remove fields unused by RAA (`stats`, `asrs`, `condition_text`, `is_architecturally_significant`, `conditions`, `satisfiable_group`, decision scores/qualities/arch_pattern_name)
3. **Structural Reshaping** — produce the `RAAInput` schema

In [7]:
def transform_arlo_to_raa_input(arlo: dict, reqs: dict) -> dict:
    """Orchestrator transformation per PRD v1 §3.2."""

    # ── §3.2.1 + §3.2.2 + §3.2.3: condition_groups ──
    condition_groups = []
    for group in arlo.get("condition_groups", []):
        cleaned_reqs = []
        for req in group.get("requirements", []):
            if not isinstance(req, dict):
                continue
            req_id = req.get("id", "")
            cleaned_reqs.append({
                "id": req_id,
                "text": reqs.get(req_id, ""),           # §3.2.1 enrichment
                "quality_attributes": req.get("quality_attributes", []),
                # condition_text and is_architecturally_significant stripped (§3.2.2)
            })
        condition_groups.append({
            "nominal_condition": group.get("nominal_condition", ""),
            "cluster": int(group.get("cluster", -1)),
            "requirements": cleaned_reqs,
            # conditions stripped (§3.2.2)
        })

    # ── §3.2.1 + §3.2.3: non_asr ──
    non_asr = [
        {"id": rid, "text": reqs.get(rid, "")}
        for rid in arlo.get("non_asr", [])
    ]

    # ── §3.2.2 + §3.2.3: concerns ──
    # Derive ccg_id: concerns map to non-foundation condition groups.
    # Use the index of non-foundation groups as the ccg_id.
    non_foundation = [
        g for g in condition_groups if g["cluster"] != -1
    ]
    concerns = []
    for i, concern in enumerate(arlo.get("concerns", [])):
        decisions = [
            {"selected_pattern": d.get("selected_pattern", "")}
            for d in concern.get("decisions", [])
            # score, satisfied_qualities, unsatisfied_qualities, arch_pattern_name stripped
        ]
        # ccg_id maps to the cluster index of non-foundation groups
        ccg_id = non_foundation[i]["cluster"] if i < len(non_foundation) else i
        concerns.append({
            "ccg_id": ccg_id,
            "weights": concern.get("weights", {}),
            "decisions": decisions,
            # satisfiable_group stripped (§3.2.2)
        })

    # ── §3.2.3: quality_weights — passed through unchanged ──
    quality_weights = arlo.get("quality_weights", {})

    return {
        "condition_groups": condition_groups,
        "concerns": concerns,
        "non_asr": non_asr,
        "quality_weights": quality_weights,
    }


raa_input = transform_arlo_to_raa_input(arlo_output, requirements)
display(Markdown("### Transformation complete — RAAInput produced"))

### Transformation complete — RAAInput produced

### 4.1 Inspect RAAInput

In [8]:
display(Markdown("#### RAAInput Summary"))
display(JSON({
    "condition_groups_count": len(raa_input["condition_groups"]),
    "concerns_count": len(raa_input["concerns"]),
    "non_asr_count": len(raa_input["non_asr"]),
    "quality_weights": raa_input["quality_weights"],
}))

display(Markdown("#### Condition Groups"))
for i, cg in enumerate(raa_input["condition_groups"]):
    display(Markdown(f"**Group {i}** — cluster={cg['cluster']}, condition=`{cg['nominal_condition'][:70]}`"))
    display(JSON({
        "requirement_count": len(cg["requirements"]),
        "requirement_ids": [r["id"] for r in cg["requirements"]],
        "sample_requirement": cg["requirements"][0] if cg["requirements"] else None,
    }))

display(Markdown("#### Concerns"))
for i, c in enumerate(raa_input["concerns"]):
    display(JSON({
        "concern_index": i,
        "ccg_id": c["ccg_id"],
        "decision_count": len(c["decisions"]),
        "sample_decisions": c["decisions"][:3],
        "weights": c["weights"],
    }))

display(Markdown("#### Non-ASR Sample"))
display(JSON(raa_input["non_asr"][:3]))

#### RAAInput Summary

<IPython.core.display.JSON object>

#### Condition Groups

**Group 0** — cluster=-1, condition=`under any circumstances`

<IPython.core.display.JSON object>

...

#### Concerns

<IPython.core.display.JSON object>

<IPython.core.display.JSON object>

#### Non-ASR Sample

<IPython.core.display.JSON object>

In [9]:
import importlib                                                                                                                                                                            
import raa                                                                                                                                                                                  
importlib.reload(raa)  

# Paste before RAA().run() call                                                                                                                                                             
from raa.embedding.embedder import assign_non_asrs                                                                                                                                          
_, assignments = assign_non_asrs(                                                                                                                                                           
    raa_input["condition_groups"],                                                                                                                                                          
    raa_input["non_asr"],                                                                                                                                                                   
    0.65,                                                                                                                                                                                   
)                                                                                                                                                                                           
from collections import Counter                                                                                                                                                             
print("Assignment distribution:", Counter(bid for _, bid in assignments))                                                                                                                   
for na, bid in assignments[:5]:                                                                                                                                                             
    print(f"  {na['id']} → {bid}")                                                                                                                                                 
                                                                                                                                                                                              

Assignment distribution: Counter({'foundation_batch': 12, 'concern_batch_3': 4, 'concern_batch_1': 2, 'concern_batch_4': 2, 'concern_batch_2': 1})
  R1 → concern_batch_1
  R2 → foundation_batch
  R3 → concern_batch_4
  R4 → concern_batch_3
  R5 → foundation_batch


## 5. Invoke RAA

Uses `RAA().run(input, config)` — the public API per `raa/main.py`.

In [10]:
from raa.main import RAA

config = {
    "configurable": {
        "thread_id": THREAD_ID,
        "db_path": DB_PATH,
        "asr_llm": llm,
        "non_asr_llm": llm,
        "judge_llm": llm,
    }
}

display(Markdown("### Invoking RAA…"))
raa_output = await RAA().run(raa_input, config)
display(Markdown("### RAA run complete ✓"))

### Invoking RAA…

Judge Step 4 — coverage structured output parsing failed for batch concern_batch_3: 1 validation error for JudgeCoverageOutput
judged
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing. Using deterministic fallback.
Judge coverage returned empty judged for 13 proposals; preserving previous judged and marking coverage unverified.


### RAA run complete ✓

## 6. Inspect RAAOutput

In [11]:
display(Markdown("### RAAOutput Overview"))
display(JSON({
    "l1_system_name": raa_output["l1_description"].get("system_name", ""),
    "l2_descriptions_count": len(raa_output["l2_descriptions"]),
    "l3_descriptions_count": len(raa_output["l3_descriptions"]),
    "entity_registry_count": len(raa_output["entity_registry"]),
    "coverage_gaps_count": len(raa_output["coverage_gaps"]),
    "unresolved_conflicts_count": len(raa_output["conflicts"]),
}))

### RAAOutput Overview

<IPython.core.display.JSON object>

### 6.1 L1 — System Context Description

In [12]:
l1 = raa_output["l1_description"]
display(Markdown(f"**System:** {l1.get('system_name', 'N/A')}"))
display(Markdown(f"**Description:** {l1.get('system_description', 'N/A')}"))
display(Markdown(f"**Boundary:** {l1.get('system_boundary_description', 'N/A')}"))
display(Markdown(f"**Actors:** {len(l1.get('actors', []))}  |  **External Systems:** {len(l1.get('external_systems', []))}  |  **Relationships:** {len(l1.get('relationships', []))}"))

if l1.get("actors"):
    display(Markdown("#### Actors"))
    display(JSON(l1["actors"]))
if l1.get("external_systems"):
    display(Markdown("#### External Systems"))
    display(JSON(l1["external_systems"]))

**System:** System

**Description:** System derived from requirements corpus.

**Boundary:** Boundary between system-owned responsibilities and external integrations.

**Actors:** 2  |  **External Systems:** 2  |  **Relationships:** 66

#### Actors

<IPython.core.display.JSON object>

#### External Systems

<IPython.core.display.JSON object>

### 6.2 L2 — Container Descriptions

In [13]:
for i, l2 in enumerate(raa_output["l2_descriptions"]):
    display(Markdown(f"#### L2[{i}] — concern_id=`{l2.get('concern_id', 'N/A')}`, condition=`{l2.get('condition', 'N/A')[:60]}`"))
    containers = l2.get("containers", [])
    display(JSON({
        "container_count": len(containers),
        "backbone_count": sum(1 for c in containers if c.get("is_backbone")),
        "concern_specific_count": sum(1 for c in containers if not c.get("is_backbone")),
        "relationship_count": len(l2.get("relationships", [])),
        "container_names": [c.get("name", "?") for c in containers],
    }))

#### L2[0] — concern_id=`concern_batch_1`, condition=`to accommodate the fluctuating number of active officers and`

<IPython.core.display.JSON object>

#### L2[1] — concern_id=`concern_batch_2`, condition=`especially in areas with higher charges`

<IPython.core.display.JSON object>

#### L2[2] — concern_id=`concern_batch_3`, condition=`when officers use public or unsecured networks`

<IPython.core.display.JSON object>

#### L2[3] — concern_id=`concern_batch_4`, condition=`During high-demand periods, such as public events or emergen`

<IPython.core.display.JSON object>

### 6.3 L3 — Component Descriptions

In [14]:
for i, l3 in enumerate(raa_output["l3_descriptions"]):
    display(Markdown(f"#### L3[{i}] — parent_container=`{l3.get('parent_container_id', 'N/A')}`, concern=`{l3.get('concern_id', 'N/A')}`"))
    components = l3.get("components", [])
    display(JSON({
        "component_count": len(components),
        "relationship_count": len(l3.get("relationships", [])),
        "component_names": [c.get("name", "?") for c in components],
    }))

#### L3[0] — parent_container=`ENT-010`, concern=`concern_batch_2`

<IPython.core.display.JSON object>

#### L3[1] — parent_container=`ENT-002`, concern=`concern_batch_4`

<IPython.core.display.JSON object>

### 6.4 Entity Registry

In [15]:
registry = raa_output["entity_registry"]
display(Markdown(f"**Total registered entities:** {len(registry)}"))

# Breakdown by c4_level and c4_type
from collections import Counter
level_counts = Counter(e.get("c4_level") for e in registry.values())
type_counts = Counter(e.get("c4_type") for e in registry.values())

display(JSON({"by_c4_level": dict(level_counts), "by_c4_type": dict(type_counts)}))

display(Markdown("#### Entity List"))
entity_summary = [
    {
        "id": e["canonical_id"],
        "name": e["canonical_name"],
        "level": e["c4_level"],
        "type": e["c4_type"],
        "authority": e["authority"],
        "source_reqs": len(e.get("source_requirements", [])),
    }
    for e in registry.values()
]
display(JSON(entity_summary))

**Total registered entities:** 55

<IPython.core.display.JSON object>

#### Entity List

<IPython.core.display.JSON object>

### 6.5 Coverage Gaps & Conflicts

In [16]:
gaps = raa_output["coverage_gaps"]
conflicts = raa_output["conflicts"]

if gaps:
    display(Markdown(f"**Coverage Gaps:** {len(gaps)}"))
    display(JSON(gaps))
else:
    display(Markdown("**Coverage Gaps:** None — full coverage achieved ✓"))

if conflicts:
    display(Markdown(f"**Unresolved Conflicts:** {len(conflicts)}"))
    display(JSON(conflicts))
else:
    display(Markdown("**Unresolved Conflicts:** None ✓"))

**Coverage Gaps:** 5

<IPython.core.display.JSON object>

**Unresolved Conflicts:** 1

<IPython.core.display.JSON object>

In [17]:
import pickle

# Save the raa_output variable to a file
with open('raa_output.pkl', 'wb') as f:
    pickle.dump(raa_output, f)
    
print("Saved raa_output to raa_output.pkl")


Saved raa_output to raa_output.pkl


In [18]:
import pickle
with open('raa_output.pkl', 'rb') as f:
    loaded_raa_output = pickle.load(f)


In [19]:
loaded_raa_output

{'l1_description': {'system_name': 'System',
  'system_description': 'System derived from requirements corpus.',
  'system_boundary_description': 'Boundary between system-owned responsibilities and external integrations.',
  'actors': [{'canonical_id': 'ENT-008',
    'name': 'Officer',
    'description': 'A law enforcement officer who interacts with the dispatch system to log incidents, request support, record statements, attach media, access laws, and receive alerts.',
    'source_requirements': ['R5', 'R7', 'R8', 'R12', 'R16', 'R20', 'R26']},
   {'canonical_id': 'ENT-044',
    'name': 'DispatchOperator',
    'description': 'A dispatch operator who monitors incident locations and officer positions on the map, sends dispatch orders, and manages communications.',
    'source_requirements': ['R2', 'R8', 'R15']}],
  'external_systems': [{'canonical_id': 'ENT-009',
    'name': 'IncidentDataSourceSystem',
    'description': 'External system or feed that provides raw incident data to the sys

In [1]:
from IPython.display import Markdown
from raa.graph import get_graph

# Get the compiled graph instance
compiled_graph = get_graph()

# Generate the mermaid text format
mermaid_code = compiled_graph.get_graph().draw_mermaid()

# Display it in the notebook output so it renders as a diagram
display(Markdown(f"```mermaid\n{mermaid_code}\n```"))

# Or if you just want to see the raw Mermaid text:
# print(mermaid_code)


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	snapshot_registry(snapshot_registry)
	asr_node(asr_node)
	non_asr_node(non_asr_node)
	judge_node(judge_node)
	next_batch(next_batch)
	assemble_output(assemble_output)
	__end__([<p>__end__</p>]):::last
	__start__ --> snapshot_registry;
	asr_node --> judge_node;
	judge_node -. &nbsp;assemble&nbsp; .-> assemble_output;
	judge_node -.-> next_batch;
	next_batch -. &nbsp;assemble&nbsp; .-> assemble_output;
	next_batch -. &nbsp;snapshot&nbsp; .-> snapshot_registry;
	non_asr_node --> judge_node;
	snapshot_registry --> asr_node;
	snapshot_registry --> non_asr_node;
	assemble_output --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```